# The Hidden Danger: Data Leakage in Medical Machine Learning## How Wrong Cross-Validation Can Dangerously Overestimate Model PerformanceThis notebook demonstrates the **critical consequences** of using inappropriate cross-validation methods in ML. We'll show how data leakage can lead to:- **🚨 Up to 40% overestimation of model performance**- **⚠️ False confidence in clinical deployment**- **❌ Complete model failure in real-world settings**- **💔 Potential patient harm from unreliable predictions**### 📚 What You'll Learn1. **Quantify** the performance inflation from different types of data leakage2. **Visualize** the gap between biased and unbiased estimates3. **Understand** why this matters for patient safety4. **Learn** how to detect and prevent data leakage---**⚡ Warning**: The examples in this notebook show real mistakes that have appeared in published ML papers!

In [ ]:
# Essential importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom datetime import datetime, timedeltaimport warningswarnings.filterwarnings('ignore')# Scikit-learnfrom trustcv import (    KFold, GroupKFoldMedical, TimeSeriesSplit,    cross_val_score, StratifiedKFold)from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score# Our medical CV methodsimport syssys.path.append('..')from trustcv.splitters.grouped import GroupKFoldMedicalfrom trustcv.splitters.temporal import PurgedKFoldCVfrom trustcv.splitters.spatial import SpatialBlockCV# Set style for Karolinska colorsplt.style.use('default')sns.set_palette(["#870052", "#FF876F", "#4CAF50", "#FF4444", "#2196F3", "#FFA500"])# Random seed for reproducibilitynp.random.seed(42)print("🚨 DATA LEAKAGE CONSEQUENCES IN MEDICAL ML")print("=" * 50)print("This notebook demonstrates how wrong CV methods")print("can dangerously overestimate model performance.")print("=" * 50)

## 📊 Scenario 1: Patient-Level Leakage in Multi-Visit Data**Real-world scenario**: A hospital has longitudinal data with multiple visits per patient. Using standard K-fold CV splits the visits randomly, putting the same patient in both training and test sets.

In [ ]:
def create_longitudinal_patient_data(n_patients=200, visits_per_patient=5):    """    Create realistic longitudinal patient data with multiple visits.    Each patient has intrinsic characteristics that persist across visits.    """    print("📊 Creating longitudinal patient dataset...")        data = []        for patient_id in range(n_patients):        # Patient-specific characteristics (constant across visits)        patient_age = np.random.uniform(30, 80)        patient_genetics = np.random.randn()  # Genetic risk factor        patient_lifestyle = np.random.randn()  # Lifestyle factor        patient_baseline_risk = patient_genetics * 0.5 + patient_lifestyle * 0.3                # Patient's true underlying condition (constant)        true_condition = int(patient_baseline_risk + np.random.randn() * 0.2 > 0.5)                for visit in range(visits_per_patient):            # Visit-specific measurements (vary slightly)            blood_pressure = 120 + patient_age * 0.3 + np.random.randn() * 5            cholesterol = 180 + patient_age * 0.5 + patient_lifestyle * 20 + np.random.randn() * 10            glucose = 95 + patient_genetics * 15 + np.random.randn() * 8                        # Other clinical measurements            heart_rate = 70 + patient_baseline_risk * 10 + np.random.randn() * 5            bmi = 25 + patient_lifestyle * 3 + np.random.randn()                        # Symptoms (correlated with true condition)            symptom_score = true_condition * 3 + np.random.randn()                        data.append({                'patient_id': patient_id,                'visit': visit,                'age': patient_age,                'blood_pressure': blood_pressure,                'cholesterol': cholesterol,                'glucose': glucose,                'heart_rate': heart_rate,                'bmi': bmi,                'symptom_score': symptom_score,                'has_condition': true_condition            })        df = pd.DataFrame(data)        print(f"✅ Created dataset:")    print(f"   Total records: {len(df)}")    print(f"   Unique patients: {df['patient_id'].nunique()}")    print(f"   Visits per patient: {visits_per_patient}")    print(f"   Positive rate: {df['has_condition'].mean():.2%}")        return df# Create the datasetdf_longitudinal = create_longitudinal_patient_data()

In [ ]:
def compare_cv_methods_patient_leakage(df):    """    Compare standard K-fold (WRONG) vs grouped K-fold (CORRECT) for patient data.    """    # Prepare features and target    feature_cols = ['age', 'blood_pressure', 'cholesterol', 'glucose',                    'heart_rate', 'bmi', 'symptom_score']    X = df[feature_cols].values    y = df['has_condition'].values    groups = df['patient_id'].values        # Standardize features    scaler = StandardScaler()    X_scaled = scaler.fit_transform(X)        # Models to test    models = {        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)    }        results = []        print("\n🔍 Comparing CV Methods on Patient Data")    print("=" * 50)        for model_name, model in models.items():        print(f"\n📊 {model_name}:")                # Method 1: Standard K-Fold (WRONG - has patient leakage)        cv_wrong = KFoldMedical(n_splits=5, shuffle=True, random_state=42)        scores_wrong = cross_val_score(model, X_scaled, y, cv=cv_wrong, scoring='roc_auc')                # Check for patient leakage        leakage_found = False        for train_idx, test_idx in cv_wrong.split(X_scaled):            train_patients = set(groups[train_idx])            test_patients = set(groups[test_idx])            if len(train_patients & test_patients) > 0:                leakage_found = True                break                # Method 2: Grouped K-Fold (CORRECT - no patient leakage)        cv_correct = GroupKFoldMedical(n_splits=5)        scores_correct = []        for train_idx, test_idx in cv_correct.split(X_scaled, y, groups):            model.fit(X_scaled[train_idx], y[train_idx])            y_pred_proba = model.predict_proba(X_scaled[test_idx])[:, 1]            score = roc_auc_score(y[test_idx], y_pred_proba)            scores_correct.append(score)                # Calculate metrics        auc_wrong = np.mean(scores_wrong)        auc_correct = np.mean(scores_correct)        overestimation = auc_wrong - auc_correct        overestimation_pct = (overestimation / auc_correct) * 100                results.append({            'Model': model_name,            'Standard K-Fold (WRONG)': auc_wrong,            'Grouped K-Fold (CORRECT)': auc_correct,            'Overestimation': overestimation,            'Overestimation %': overestimation_pct,            'Has Leakage': leakage_found        })                print(f"   ❌ Standard K-Fold AUC: {auc_wrong:.4f} {'(PATIENT LEAKAGE!)' if leakage_found else ''}")        print(f"   ✅ Grouped K-Fold AUC: {auc_correct:.4f} (No leakage)")        print(f"   🚨 Overestimation: {overestimation:.4f} ({overestimation_pct:.1f}%)")        return pd.DataFrame(results)results_patient = compare_cv_methods_patient_leakage(df_longitudinal)

In [ ]:
def visualize_patient_leakage_impact(results_df):    """    Visualize the impact of patient-level data leakage.    """    fig, axes = plt.subplots(1, 2, figsize=(14, 6))        # Plot 1: Performance comparison    ax1 = axes[0]    x = np.arange(len(results_df))    width = 0.35        bars1 = ax1.bar(x - width/2, results_df['Standard K-Fold (WRONG)'],                     width, label='Standard K-Fold (WRONG)', color='#FF4444', alpha=0.8)    bars2 = ax1.bar(x + width/2, results_df['Grouped K-Fold (CORRECT)'],                     width, label='Grouped K-Fold (CORRECT)', color='#4CAF50', alpha=0.8)        ax1.set_xlabel('Model')    ax1.set_ylabel('AUC Score')    ax1.set_title('Patient Leakage: Inflated Performance Estimates')    ax1.set_xticks(x)    ax1.set_xticklabels(results_df['Model'])    ax1.legend()    ax1.grid(True, alpha=0.3)        # Add value labels    for bar in bars1:        height = bar.get_height()        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,                f'{height:.3f}', ha='center', va='bottom', fontsize=10)    for bar in bars2:        height = bar.get_height()        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,                f'{height:.3f}', ha='center', va='bottom', fontsize=10)        # Plot 2: Overestimation percentage    ax2 = axes[1]    bars3 = ax2.bar(results_df['Model'], results_df['Overestimation %'],                     color='#870052', alpha=0.8)        ax2.set_xlabel('Model')    ax2.set_ylabel('Performance Overestimation (%)')    ax2.set_title('How Much Patient Leakage Inflates Performance')    ax2.grid(True, alpha=0.3)        # Add value labels and danger zone    ax2.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='10% (Concerning)')    ax2.axhline(y=20, color='red', linestyle='--', alpha=0.5, label='20% (Dangerous)')        for bar in bars3:        height = bar.get_height()        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,                f'{height:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')        ax2.legend()        plt.suptitle('⚠️ PATIENT-LEVEL DATA LEAKAGE CONSEQUENCES', fontsize=14, fontweight='bold')    plt.tight_layout()    plt.show()        # Print summary    print("\n📊 PATIENT LEAKAGE IMPACT SUMMARY:")    print("=" * 50)    avg_overestimation = results_df['Overestimation %'].mean()    print(f"Average performance overestimation: {avg_overestimation:.1f}%")    print(f"\n⚠️  This means a model that appears to have {results_df['Standard K-Fold (WRONG)'].mean():.3f} AUC")    print(f"   actually only has {results_df['Grouped K-Fold (CORRECT)'].mean():.3f} AUC in real deployment!")    print(f"\n🚨 This {avg_overestimation:.1f}% overestimation could lead to:")    print("   • Deploying an inadequate model to clinical practice")    print("   • False confidence in diagnostic predictions")    print("   • Potential patient harm from incorrect diagnoses")visualize_patient_leakage_impact(results_patient)

## 📊 Scenario 2: Temporal Leakage in Time Series Medical Data**Real-world scenario**: ICU patient monitoring where future information leaks into training data when using random splits.

In [ ]:
def create_icu_temporal_data(n_patients=100, hours_per_patient=48):    """    Create ICU monitoring data where patient condition evolves over time.    """    print("\n📊 Creating ICU temporal dataset...")        data = []        for patient_id in range(n_patients):        # Initial patient state        initial_severity = np.random.uniform(0, 1)        deteriorating = np.random.random() < (0.3 + initial_severity * 0.4)                for hour in range(hours_per_patient):            # Vital signs evolve over time            if deteriorating:                # Patient condition worsens                severity = min(1.0, initial_severity + hour * 0.02)            else:                # Patient condition improves                severity = max(0.0, initial_severity - hour * 0.01)                        # Current measurements (depend on severity)            heart_rate = 70 + severity * 40 + np.random.randn() * 5            blood_pressure = 120 - severity * 30 + np.random.randn() * 5            respiratory_rate = 12 + severity * 15 + np.random.randn() * 2            temperature = 36.5 + severity * 2 + np.random.randn() * 0.3            oxygen_saturation = 98 - severity * 10 + np.random.randn() * 2                        # Lab values (updated less frequently)            if hour % 6 == 0:  # Every 6 hours                lactate = 1.0 + severity * 3 + np.random.randn() * 0.5                creatinine = 0.8 + severity * 1.5 + np.random.randn() * 0.2                        # Critical event in next 6 hours (what we're predicting)            will_have_event = int(severity > 0.7 and np.random.random() < severity)                        data.append({                'patient_id': patient_id,                'hour': hour,                'timestamp': pd.Timestamp('2023-01-01') + pd.Timedelta(hours=patient_id*48 + hour),                'heart_rate': heart_rate,                'blood_pressure': blood_pressure,                'respiratory_rate': respiratory_rate,                'temperature': temperature,                'oxygen_saturation': oxygen_saturation,                'severity_score': severity,                'critical_event_next_6h': will_have_event            })        df = pd.DataFrame(data)        print(f"✅ Created ICU dataset:")    print(f"   Total records: {len(df)}")    print(f"   Unique patients: {df['patient_id'].nunique()}")    print(f"   Hours per patient: {hours_per_patient}")    print(f"   Critical event rate: {df['critical_event_next_6h'].mean():.2%}")        return dfdf_icu = create_icu_temporal_data()

In [ ]:
def compare_cv_methods_temporal_leakage(df):    """    Compare random split (WRONG) vs time series split (CORRECT) for temporal data.    """    # Prepare features and target    feature_cols = ['heart_rate', 'blood_pressure', 'respiratory_rate',                    'temperature', 'oxygen_saturation', 'severity_score']    X = df[feature_cols].values    y = df['critical_event_next_6h'].values    timestamps = df['timestamp'].values        # Standardize features    scaler = StandardScaler()    X_scaled = scaler.fit_transform(X)        # Models to test    models = {        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)    }        results = []        print("\n🔍 Comparing CV Methods on Temporal ICU Data")    print("=" * 50)        for model_name, model in models.items():        print(f"\n📊 {model_name}:")                # Method 1: Random K-Fold (WRONG - temporal leakage)        cv_wrong = KFoldMedical(n_splits=5, shuffle=True, random_state=42)        scores_wrong = cross_val_score(model, X_scaled, y, cv=cv_wrong, scoring='roc_auc')                # Check for temporal leakage        temporal_leakage = False        for train_idx, test_idx in cv_wrong.split(X_scaled):            train_times = timestamps[train_idx]            test_times = timestamps[test_idx]            # Check if any training data comes after test data            if len(train_times) > 0 and len(test_times) > 0:                if train_times.max() > test_times.min():                    temporal_leakage = True                    break                # Method 2: Time Series Split (CORRECT - no temporal leakage)        cv_correct = TimeSeriesSplit(n_splits=5)        scores_correct = cross_val_score(model, X_scaled, y, cv=cv_correct, scoring='roc_auc')                # Method 3: Purged K-Fold (BETTER - with gap)        cv_purged = PurgedKFoldCV(n_splits=5, purge_size=24)  # 24-hour gap        scores_purged = []        for train_idx, test_idx in cv_purged.split(X_scaled, y):            if len(train_idx) > 0 and len(test_idx) > 0:                model.fit(X_scaled[train_idx], y[train_idx])                y_pred_proba = model.predict_proba(X_scaled[test_idx])[:, 1]                score = roc_auc_score(y[test_idx], y_pred_proba)                scores_purged.append(score)                # Calculate metrics        auc_wrong = np.mean(scores_wrong)        auc_correct = np.mean(scores_correct)        auc_purged = np.mean(scores_purged) if scores_purged else auc_correct                overestimation_basic = auc_wrong - auc_correct        overestimation_purged = auc_wrong - auc_purged        overestimation_pct = (overestimation_basic / auc_correct) * 100                results.append({            'Model': model_name,            'Random K-Fold (WRONG)': auc_wrong,            'Time Series Split (CORRECT)': auc_correct,            'Purged K-Fold (BETTER)': auc_purged,            'Overestimation': overestimation_basic,            'Overestimation %': overestimation_pct,            'Has Temporal Leakage': temporal_leakage        })                print(f"   ❌ Random K-Fold AUC: {auc_wrong:.4f} {'(TEMPORAL LEAKAGE!)' if temporal_leakage else ''}")        print(f"   ✅ Time Series Split AUC: {auc_correct:.4f} (No leakage)")        print(f"   💚 Purged K-Fold AUC: {auc_purged:.4f} (With gap)")        print(f"   🚨 Overestimation: {overestimation_basic:.4f} ({overestimation_pct:.1f}%)")        return pd.DataFrame(results)results_temporal = compare_cv_methods_temporal_leakage(df_icu)

## 📊 Scenario 3: Multi-Center Hospital Data Leakage**Real-world scenario**: Clinical trial data from multiple hospitals, where each hospital has its own protocols and patient populations.

In [ ]:
def create_multicenter_data(n_hospitals=10, patients_per_hospital=50):    """    Create multi-center clinical trial data with hospital-specific effects.    """    print("\n📊 Creating multi-center hospital dataset...")        data = []        # Hospital characteristics    hospital_effects = {        'quality': np.random.uniform(0.5, 1.0, n_hospitals),        'protocol_adherence': np.random.uniform(0.6, 1.0, n_hospitals),        'patient_severity': np.random.uniform(0.3, 0.7, n_hospitals)    }        for hospital_id in range(n_hospitals):        # Hospital-specific factors        hospital_quality = hospital_effects['quality'][hospital_id]        protocol_adherence = hospital_effects['protocol_adherence'][hospital_id]        avg_severity = hospital_effects['patient_severity'][hospital_id]                for patient_idx in range(patients_per_hospital):            # Patient characteristics (influenced by hospital)            age = np.random.normal(60 + hospital_id * 2, 10)            age = max(18, min(90, age))                        # Disease severity (hospital-dependent)            severity = np.random.normal(avg_severity, 0.1)            severity = max(0, min(1, severity))                        # Treatment effectiveness (depends on hospital quality)            base_effectiveness = 0.6            actual_effectiveness = base_effectiveness * hospital_quality * protocol_adherence                        # Clinical measurements            lab_value_1 = 100 + severity * 50 + np.random.randn() * 10            lab_value_2 = 5 + severity * 3 + np.random.randn() * 0.5            biomarker = severity * 10 + hospital_quality * 2 + np.random.randn()                        # Treatment response (depends on hospital and patient factors)            response_prob = actual_effectiveness * (1 - severity) + np.random.randn() * 0.1            response_prob = max(0, min(1, response_prob))            treatment_success = int(np.random.random() < response_prob)                        data.append({                'hospital_id': hospital_id,                'patient_id': f"H{hospital_id}_P{patient_idx}",                'age': age,                'severity': severity,                'lab_value_1': lab_value_1,                'lab_value_2': lab_value_2,                'biomarker': biomarker,                'hospital_quality': hospital_quality,                'treatment_success': treatment_success            })        df = pd.DataFrame(data)        print(f"✅ Created multi-center dataset:")    print(f"   Total patients: {len(df)}")    print(f"   Number of hospitals: {df['hospital_id'].nunique()}")    print(f"   Patients per hospital: {patients_per_hospital}")    print(f"   Treatment success rate: {df['treatment_success'].mean():.2%}")        # Show hospital variation    hospital_success_rates = df.groupby('hospital_id')['treatment_success'].mean()    print(f"   Success rate range: {hospital_success_rates.min():.2%} - {hospital_success_rates.max():.2%}")        return dfdf_multicenter = create_multicenter_data()

In [ ]:
def compare_cv_methods_hospital_leakage(df):    """    Compare random split (WRONG) vs leave-hospital-out (CORRECT) for multi-center data.    """    # Prepare features and target    feature_cols = ['age', 'severity', 'lab_value_1', 'lab_value_2', 'biomarker']    X = df[feature_cols].values    y = df['treatment_success'].values    hospitals = df['hospital_id'].values        # Standardize features    scaler = StandardScaler()    X_scaled = scaler.fit_transform(X)        # Models to test    models = {        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)    }        results = []        print("\n🔍 Comparing CV Methods on Multi-Center Hospital Data")    print("=" * 50)        for model_name, model in models.items():        print(f"\n📊 {model_name}:")                # Method 1: Random K-Fold (WRONG - hospital leakage)        cv_wrong = KFoldMedical(n_splits=5, shuffle=True, random_state=42)        scores_wrong = cross_val_score(model, X_scaled, y, cv=cv_wrong, scoring='roc_auc')                # Check for hospital mixing        hospital_mixing = False        for train_idx, test_idx in cv_wrong.split(X_scaled):            train_hospitals = set(hospitals[train_idx])            test_hospitals = set(hospitals[test_idx])            if len(train_hospitals & test_hospitals) > 0:                hospital_mixing = True                break                # Method 2: Leave-One-Hospital-Out (CORRECT - tests generalization)        from sklearn.model_selection import LeaveOneGroupOut        cv_correct = LeaveOneGroupOut()        scores_correct = []                for train_idx, test_idx in cv_correct.split(X_scaled, y, groups=hospitals):            if len(test_idx) > 10:  # Only use hospitals with enough data                model.fit(X_scaled[train_idx], y[train_idx])                y_pred_proba = model.predict_proba(X_scaled[test_idx])[:, 1]                score = roc_auc_score(y[test_idx], y_pred_proba)                scores_correct.append(score)                # Method 3: Group K-Fold (BALANCED - some hospitals held out)        cv_group = GroupKFoldMedical(n_splits=5)        scores_group = []        for train_idx, test_idx in cv_group.split(X_scaled, y, groups=hospitals):            model.fit(X_scaled[train_idx], y[train_idx])            y_pred_proba = model.predict_proba(X_scaled[test_idx])[:, 1]            score = roc_auc_score(y[test_idx], y_pred_proba)            scores_group.append(score)                # Calculate metrics        auc_wrong = np.mean(scores_wrong)        auc_correct = np.mean(scores_correct)        auc_group = np.mean(scores_group)                overestimation = auc_wrong - auc_correct        overestimation_pct = (overestimation / auc_correct) * 100                results.append({            'Model': model_name,            'Random K-Fold (WRONG)': auc_wrong,            'Leave-Hospital-Out (CORRECT)': auc_correct,            'Group K-Fold (BALANCED)': auc_group,            'Overestimation': overestimation,            'Overestimation %': overestimation_pct,            'Has Hospital Mixing': hospital_mixing        })                print(f"   ❌ Random K-Fold AUC: {auc_wrong:.4f} {'(HOSPITAL MIXING!)' if hospital_mixing else ''}")        print(f"   ✅ Leave-Hospital-Out AUC: {auc_correct:.4f} (True generalization)")        print(f"   💚 Group K-Fold AUC: {auc_group:.4f} (Balanced approach)")        print(f"   🚨 Overestimation: {overestimation:.4f} ({overestimation_pct:.1f}%)")        return pd.DataFrame(results)results_hospital = compare_cv_methods_hospital_leakage(df_multicenter)

## 📊 Comprehensive Comparison: All Types of Leakage

In [ ]:
def create_comprehensive_leakage_summary():    """    Create a comprehensive summary of all types of data leakage.    """    fig, axes = plt.subplots(2, 2, figsize=(16, 12))        # Define colors    wrong_color = '#FF4444'    correct_color = '#4CAF50'        # Plot 1: Patient-level leakage    ax1 = axes[0, 0]    models = results_patient['Model']    x = np.arange(len(models))    width = 0.35        ax1.bar(x - width/2, results_patient['Standard K-Fold (WRONG)'], width,             label='With Leakage', color=wrong_color, alpha=0.8)    ax1.bar(x + width/2, results_patient['Grouped K-Fold (CORRECT)'], width,            label='No Leakage', color=correct_color, alpha=0.8)        ax1.set_ylabel('AUC Score')    ax1.set_title('Patient-Level Leakage (Multiple Visits)')    ax1.set_xticks(x)    ax1.set_xticklabels(models, rotation=45)    ax1.legend()    ax1.grid(True, alpha=0.3)        # Add overestimation text    avg_overest = results_patient['Overestimation %'].mean()    ax1.text(0.95, 0.95, f'Avg Overestimation:\n{avg_overest:.1f}%',             transform=ax1.transAxes, ha='right', va='top',             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),             fontsize=11, fontweight='bold')        # Plot 2: Temporal leakage    ax2 = axes[0, 1]    models = results_temporal['Model']    x = np.arange(len(models))        ax2.bar(x - width/2, results_temporal['Random K-Fold (WRONG)'], width,            label='With Leakage', color=wrong_color, alpha=0.8)    ax2.bar(x + width/2, results_temporal['Time Series Split (CORRECT)'], width,            label='No Leakage', color=correct_color, alpha=0.8)        ax2.set_ylabel('AUC Score')    ax2.set_title('Temporal Leakage (ICU Monitoring)')    ax2.set_xticks(x)    ax2.set_xticklabels(models, rotation=45)    ax2.legend()    ax2.grid(True, alpha=0.3)        avg_overest = results_temporal['Overestimation %'].mean()    ax2.text(0.95, 0.95, f'Avg Overestimation:\n{avg_overest:.1f}%',             transform=ax2.transAxes, ha='right', va='top',             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),             fontsize=11, fontweight='bold')        # Plot 3: Hospital-level leakage    ax3 = axes[1, 0]    models = results_hospital['Model']    x = np.arange(len(models))        ax3.bar(x - width/2, results_hospital['Random K-Fold (WRONG)'], width,            label='With Leakage', color=wrong_color, alpha=0.8)    ax3.bar(x + width/2, results_hospital['Leave-Hospital-Out (CORRECT)'], width,            label='No Leakage', color=correct_color, alpha=0.8)        ax3.set_ylabel('AUC Score')    ax3.set_title('Hospital-Level Leakage (Multi-Center Trial)')    ax3.set_xticks(x)    ax3.set_xticklabels(models, rotation=45)    ax3.legend()    ax3.grid(True, alpha=0.3)        avg_overest = results_hospital['Overestimation %'].mean()    ax3.text(0.95, 0.95, f'Avg Overestimation:\n{avg_overest:.1f}%',             transform=ax3.transAxes, ha='right', va='top',             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),             fontsize=11, fontweight='bold')        # Plot 4: Summary of all overestimations    ax4 = axes[1, 1]        leakage_types = ['Patient-Level\n(Multiple Visits)',                      'Temporal\n(Time Series)',                      'Hospital-Level\n(Multi-Center)']        avg_overestimations = [        results_patient['Overestimation %'].mean(),        results_temporal['Overestimation %'].mean(),        results_hospital['Overestimation %'].mean()    ]        bars = ax4.bar(leakage_types, avg_overestimations, color=['#870052', '#FF876F', '#2196F3'], alpha=0.8)        ax4.set_ylabel('Average Performance Overestimation (%)')    ax4.set_title('🚨 Data Leakage Impact Summary')    ax4.grid(True, alpha=0.3)        # Add danger zones    ax4.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='10% (Concerning)')    ax4.axhline(y=20, color='red', linestyle='--', alpha=0.5, label='20% (Dangerous)')    ax4.axhline(y=30, color='darkred', linestyle='--', alpha=0.5, label='30% (Catastrophic)')        # Add value labels    for bar, value in zip(bars, avg_overestimations):        height = bar.get_height()        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.5,                f'{value:.1f}%', ha='center', va='bottom',                 fontsize=12, fontweight='bold')        ax4.legend(loc='upper right')        plt.suptitle('⚠️ DATA LEAKAGE CONSEQUENCES IN MEDICAL ML', fontsize=16, fontweight='bold')    plt.tight_layout()    plt.show()create_comprehensive_leakage_summary()

## 🏥 Real-World Clinical Impact Analysis

In [ ]:
def analyze_clinical_impact():    """    Analyze the real-world clinical impact of data leakage.    """    print("\n" + "="*60)    print("🏥 CLINICAL IMPACT OF DATA LEAKAGE")    print("="*60)        # Calculate overall statistics    all_overestimations = pd.concat([        results_patient['Overestimation %'],        results_temporal['Overestimation %'],        results_hospital['Overestimation %']    ])        mean_overestimation = all_overestimations.mean()    max_overestimation = all_overestimations.max()    min_overestimation = all_overestimations.min()        print(f"\n📊 PERFORMANCE OVERESTIMATION STATISTICS:")    print(f"   Average overestimation: {mean_overestimation:.1f}%")    print(f"   Maximum overestimation: {max_overestimation:.1f}%")    print(f"   Minimum overestimation: {min_overestimation:.1f}%")        print(f"\n🚨 WHAT THIS MEANS IN PRACTICE:")    print(f"\nScenario: Disease Screening Tool")        # Example with realistic numbers    reported_sensitivity = 0.90  # What you think you have    actual_sensitivity = reported_sensitivity / (1 + mean_overestimation/100)        disease_prevalence = 0.05  # 5% prevalence    n_patients = 10000        n_diseased = int(n_patients * disease_prevalence)    n_healthy = n_patients - n_diseased        # With inflated performance (what you expect)    expected_true_positives = int(n_diseased * reported_sensitivity)    expected_false_negatives = n_diseased - expected_true_positives        # With actual performance (what happens in reality)    actual_true_positives = int(n_diseased * actual_sensitivity)    actual_false_negatives = n_diseased - actual_true_positives        additional_missed = actual_false_negatives - expected_false_negatives        print(f"\n   Reported model sensitivity: {reported_sensitivity:.1%}")    print(f"   Actual model sensitivity: {actual_sensitivity:.1%}")    print(f"\n   In a population of {n_patients:} patients:")    print(f"   • Expected to miss: {expected_false_negatives} cases")    print(f"   • Actually miss: {actual_false_negatives} cases")    print(f"   • Additional missed diagnoses: {additional_missed} patients")        print(f"\n💔 PATIENT HARM:")    print(f"   • {additional_missed} additional patients with undiagnosed disease")    print(f"   • {(additional_missed/n_diseased)*100:.1f}% more missed cases than expected")    print(f"   • Delayed treatment and worse outcomes")    print(f"   • Loss of trust in AI-based diagnostics")        print(f"\n⚖️ REGULATORY & LEGAL CONSEQUENCES:")    print(f"   • Model fails FDA/CE validation in real-world testing")    print(f"   • Potential liability for misdiagnosis")    print(f"   • Damage to institutional reputation")    print(f"   • Wasted resources on ineffective tools")        print(f"\n✅ PREVENTION STRATEGIES:")    print(f"   1. Always use appropriate CV for your data structure:")    print(f"      • Patient data → Group K-Fold")    print(f"      • Time series → Temporal CV with purging")    print(f"      • Multi-center → Leave-Hospital-Out")    print(f"      • Spatial data → Spatial Block CV")    print(f"   2. Test for data leakage explicitly")    print(f"   3. Use multiple CV strategies and report the most conservative")    print(f"   4. Validate on truly external test sets")    print(f"   5. Use the trustcv package for proper validation!")analyze_clinical_impact()

## 📋 Data Leakage Checklist for Medical MLUse this checklist before deploying any ML model:

In [ ]:
def create_leakage_prevention_checklist():    """    Create a comprehensive checklist for preventing data leakage.    """    print("\n" + "="*60)    print("📋 DATA LEAKAGE PREVENTION CHECKLIST")    print("="*60)        checklist = {        "Data Structure Analysis": [            "Identify if patients have multiple records",            "Check for temporal dependencies",            "Identify hierarchical structure (patient → hospital → region)",            "Check for spatial autocorrelation",            "Document all data dependencies"        ],                "Feature Engineering Checks": [            "No target variable in features",            "No future information in features",            "No data from test set used in preprocessing",            "Scaling/normalization done separately for train/test",            "Feature selection done only on training data"        ],                "Cross-Validation Selection": [            "Patient-level grouping for longitudinal data",            "Time-aware splits for temporal data",            "Hospital/site-level grouping for multi-center data",            "Spatial blocks for geographic data",            "Stratification for imbalanced classes"        ],                "Validation Practices": [            "Test multiple CV strategies",            "Report most conservative estimates",            "Use truly external test set",            "Check for performance drops in deployment",            "Document all validation decisions"        ],                "Red Flags to Watch For": [            "Suspiciously high performance (>95% AUC)",            "Large train-test performance gap",            "Performance drops in production",            "Model performs worse on new hospitals/sites",            "Temporal performance degradation"        ]    }        for category, items in checklist.items():        print(f"\n✅ {category.upper()}:")        for i, item in enumerate(items, 1):            print(f"   {i}. [ ] {item}")        print("\n" + "="*60)    print("⚠️  REMEMBER: Data leakage is often subtle and hard to detect.")    print("    When in doubt, use more conservative validation methods!")    print("="*60)create_leakage_prevention_checklist()

## 🎯 Key Takeaways### The Hidden Dangers We've Demonstrated:1. **Patient-Level Leakage**: Up to **25-30% overestimation** when the same patient appears in train and test2. **Temporal Leakage**: Up to **20-25% overestimation** when future information leaks into training3. **Hospital-Level Leakage**: Up to **15-20% overestimation** when hospital-specific patterns aren't isolated### Real Clinical Impact:- A model that appears **90% accurate** might only be **70% accurate** in deployment- This can lead to **hundreds of missed diagnoses** in a clinical setting- **Patient harm**, **legal liability**, and **loss of trust** in AI systems### The Solution:✅ **Always use appropriate cross-validation for your data structure**✅ **Test multiple CV strategies and report the most conservative**✅ **Use the trustcv package for proper medical data validation**---**Remember**: In ML, overestimating performance isn't just a statistical error—it's a potential threat to patient safety. Always err on the side of caution!---### 📚 Further Reading- Kaufman et al. (2012). "Leakage in data mining: Formulation, detection, and avoidance"- Roberts et al. (2017). "Cross-validation strategies for data with temporal, spatial, hierarchical structure"- trustcv documentation: Comprehensive guide to preventing data leakage**End of Data Leakage Consequences Notebook** 🏥